# 09.03 · Handoffs

**Module 09 · Multi-Agent Systems** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lesson 09.02; Module 04 (`@wrap_model_call`); Module 07 (`Command`). |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for the 💸 cell) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Build** state-driven handoffs in **one** agent: middleware swaps the system prompt and tools based on `current_step` (the form the docs recommend).
2. **Build** handoffs between **agent subgraphs**: `Command(goto=..., graph=Command.PARENT)` and an `active_agent` key (triage → billing → technical).
3. **Explain** why the next turn skips triage, and measure what that saves vs subagents.
4. **Debug** the classic handoff bug: lost context and agents handing back and forth until the recursion limit.
5. **Keep** human approval working inside a handoff (refunds over $50).

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Handoffs](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs) | Single agent with middleware vs multiple agent subgraphs; `Command.PARENT`; keeping AI/Tool message pairs valid; context engineering |
| [Build customer support with handoffs](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs-customer-support) | A step-by-step state machine (`current_step`) with `@wrap_model_call` |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/09-multi-agent-systems/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Offline: the shared desk (Lesson 09.01 Part 2).
from langgraph.checkpoint.memory import InMemorySaver
from mastery.agents import UsageMeter
from mastery.agents.support import (DESCRIPTIONS, DOMAINS, TRIAGE_PROMPT, SupportDesk, evaluate, faq_search,
                                    load_conversations, make_specialists, make_support_tools, offline_model,
                                    run_conversation)

desk = SupportDesk.load()
tools = make_support_tools(desk, faq_search())
by_id = {c["id"]: c for c in load_conversations()}
print(TRIAGE_PROMPT)

**You should now see** the triage prompt: it never solves problems itself, it transfers to billing, technical or account, or asks one short question.

---
## Part 1 · Handoffs inside ONE agent (middleware + `current_step`)

The docs' recommended form: one `create_agent` with **all** tools registered, a `current_step` key in its state, and `@wrap_model_call` middleware that, before each model call, sets the step's system prompt and narrows the tools. A transfer tool just updates `current_step` (and answers its tool call with a `ToolMessage`). `mastery.agents.step_tool` and `step_middleware` are those two pieces.

In [ ]:
# 🧪 Offline: triage → specialist as steps of one agent.
from langchain.agents import create_agent
from mastery.agents import StepState, step_middleware, step_tool

specialists = make_specialists(tools, handoffs=True)       # prompts with "transfer when needed"
transfers = {d: step_tool(d, description=f"Transfer to the {d} specialist: {DESCRIPTIONS[d]}") for d in DOMAINS}
steps = {"triage": {"prompt": TRIAGE_PROMPT, "tools": list(transfers.values())}}
for d, spec in specialists.items():
    steps[d] = {"prompt": spec.system_prompt, "tools": [*spec.tools, *(t for o, t in transfers.items() if o != d)]}

one_agent = create_agent(offline_model(), tools=[*transfers.values(), *(t for d in DOMAINS for t in tools[d])],
                         middleware=[step_middleware(steps, default="triage")], state_schema=StepState,
                         checkpointer=InMemorySaver(), name="support_desk")
config = {"configurable": {"thread_id": "steps-1"}}
for turn in by_id["S12"]["turns"]:
    out = one_agent.invoke({"messages": [{"role": "user", "content": turn}]}, config)
    print(f"step={out['current_step']:<8} | {out['messages'][-1].content[:90]}")
print({name: [t if isinstance(t, str) else t.name for t in cfg["tools"]] for name, cfg in steps.items()}["billing"])

**You should now see** two lines, both `step=billing`: the invoice list (`Invoices for C-1001 (Pro plan): INV-1001 ...`), then `Refunded $49.00 for INV-1003 ...`; and the billing step's tools: `lookup_invoices`, `issue_refund`, `transfer_to_technical`, `transfer_to_account`.

### What just happened?

Turn 1: the model ran with the **triage** prompt and only the 3 transfer tools; it called `transfer_to_billing`, which set `current_step="billing"`. The next model call (same turn) got the **billing** prompt and tools. Turn 2 started in `billing` because `current_step` is saved in the checkpoint: **no triage call** on repeat requests. That's why handoffs need 5 calls for S12 where subagents need 8 (Lesson 09.01).

---
## Part 2 · Handoffs between agent subgraphs (`Command.PARENT`)

When specialists need to be **separate agents** (own middleware, own state, built by another team), make each one a node of a parent graph. A transfer tool returns `Command(goto="billing", graph=Command.PARENT, update={"active_agent": "billing", "messages": [...]})`: it jumps in the **parent** graph. A new user turn starts at `active_agent`. `handoff_tool` and `handoff_graph` package that.

In [ ]:
# 🧪 Offline: triage + 3 specialist agents as graph nodes; watch who works on each turn of S13.
from mastery.agents.support import build_handoffs_desk

graph_desk = build_handoffs_desk(offline_model, tools, checkpointer=InMemorySaver())
print("nodes:", [n for n in graph_desk.get_graph().nodes if not n.startswith("__")])
desk.reset()
config = {"configurable": {"thread_id": "graph-1"}}
for turn in by_id["S13"]["turns"]:
    meter = UsageMeter()
    out = graph_desk.invoke({"messages": [{"role": "user", "content": turn}]}, {**config, "callbacks": [meter]})
    print(f"active={out['active_agent']:<9} calls={meter.agents()} | {out['messages'][-1].content[:70]}")

**You should now see** `nodes: ['triage', 'billing', 'technical', 'account']`, then turn 1 with `active=billing` and calls by `triage` (1) and `billing` (2), and turn 2 with `active=technical` and calls by `billing` (1: it sees a technical question and transfers) and `technical` (2), ending with two FAQ passages: `Failed payments` (billing FAQ) first, then `Webhooks` (`retried 5 times over 24 hours`). BM25 matched *fail* and *retr...* in the billing FAQ too; the answer still contains the fact, but a hybrid or filtered search (Module 06) would rank the right passage first.

### What just happened?

Turn 2 started at **billing** (the active agent), not triage. Billing's prompt says "transfer when the latest message needs another specialist", so it handed over. Each agent node runs `agent.invoke(state)` on the **shared** message list; `handoff_graph` ends the turn when an agent answers without tool calls.

---
## Part 3 · The handoff bug: what does the next agent see?

The docs' example passes only the **last AI message** (the one with the transfer call) plus a `ToolMessage` answering it. That keeps the message history valid (every tool call has a result), but **everything else the agent did in that turn stays inside its subgraph**. `handoff_tool(carry="last")` does that; `carry="turn"` (the desk's default) passes every message since the user's last message.

S14 needs billing **and** technical in one turn. Watch what happens with `"last"`.

In [ ]:
# 🧪 Offline: S14 with carry="last" vs carry="turn" (recursion_limit=40 per turn, loop guard OFF to see the bug).
rows = []
for carry in ("last", "turn"):
    meter = UsageMeter()
    app = build_handoffs_desk(offline_model, tools, checkpointer=InMemorySaver(), carry=carry, max_handoffs=None)
    report = evaluate(app, [by_id["S14"]], desk, meter=meter)
    o = report["rows"][0]["outcome"]
    rows.append({"carry": carry, "passed": report["passed"], "model calls": meter.model_calls,
                 "tools logged": len(o["tools"]), "answer": o["answer"][:60]})
table(rows, ["carry", "passed", "model calls", "tools logged", "answer"])

**You should now see** `last`: not passed, **79** model calls, about 40 tools logged, `⚠️ GraphRecursionError: Recursion limit of 40 reached ...`; `turn`: passed with **5** calls and 2 tools.

### What just happened?

With `"last"`, billing refunded, then transferred to technical, but its `issue_refund` result never reached the shared state. Technical answered the 429 part, saw no billing result, and (following its prompt) transferred back to billing, which refunded again ("already refunded"), and so on: a **ping-pong** until `recursion_limit` stopped it. A real LLM does a softer version of this (repeating a question, re-checking an invoice). Two fixes: summarize the work done into the transfer `ToolMessage` (what the handoffs docs' context-engineering section recommends: pass what the next agent needs, not everything), or pass the whole turn (`carry="turn"`, this course's shortcut: simple, but the next agent's context grows with every tool result). And never rely on `recursion_limit` alone: it stopped this run only after **79** model calls and about 20 refund attempts.

### A loop guard that escalates

A ping-pong is easy to detect: the same user turn keeps producing transfers. `build_handoffs_desk(max_handoffs=4)` (the default) counts `transfer_to_*` handoffs since the latest user message; one more than that and the next agent node answers *"I'm passing this conversation to a human support agent ..."* instead of running. Same bug, guard on:

In [ ]:
# 🧪 Offline: the same carry="last" bug, with the default loop guard (max_handoffs=4).
meter = UsageMeter()
app = build_handoffs_desk(offline_model, tools, checkpointer=InMemorySaver(), carry="last", max_handoffs=4)
report = evaluate(app, [by_id["S14"]], desk, meter=meter)
o = report["rows"][0]["outcome"]
print("escalated:", o["escalated"], "| model calls:", meter.model_calls, "| tools logged:", len(o["tools"]))
print("answer:", o["answer"])

**You should now see** `escalated: True` with about a dozen model calls instead of 79, and the escalation message as the answer. The conversation still fails the test (the customer didn't get their answer), but it fails **fast, cheaply and visibly**: `outcome["escalated"]` is something an eval or a dashboard can count.

> ⚠️ **Production note: a loop guard is not a fix.**
> `max_handoffs` caps the damage of a ping-pong; it doesn't remove the cause. In production: alert on escalations per hour, route them to a real human queue (Module 08's interrupts, or a ticket), and fix the context bug the escalations reveal.

---
## Part 4 · Human approval inside a handoff

In [ ]:
# 🧪 Offline: a $120 partial refund (S03): billing pauses inside the graph; the manager rejects.
from langgraph.types import Command

desk.reset()
app = build_handoffs_desk(offline_model, tools, checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "refund-120"}}
out = app.invoke({"messages": [{"role": "user", "content": by_id["S03"]["turns"][0]}]}, config)
print("paused at:", app.get_state(config).next, "|", out["__interrupt__"][0].value["question"])
out = app.invoke(Command(resume={"action": "reject", "feedback": "partial-month refunds are not offered"}), config)
print("answer:", out["messages"][-1].content)
print("refunds issued:", desk.refunds)

**You should now see** `paused at: ('billing',)` and the question `Approve a $120.00 refund (over $50)? approve / edit (new amount) / reject`, then `Refund of $120.00 for INV-2001 was not approved by a billing manager (partial-month refunds are not offered).` and `refunds issued: []`.

Here the pause **is** visible as a graph node (`billing`), unlike subagents inside tools (09.02). The resume values are the course's `parse_decision` format from Lesson 08.03: `"approve"`, `{"action": "edit", "text": "60"}` (a new amount) or `{"action": "reject", "feedback": ...}`.

---
## Part 5 · Both handoff forms on the test set

In [ ]:
# 🧪 Offline: one-agent steps vs agent subgraphs, 15 conversations.
from mastery.agents.support import build_single_agent_desk

rows = []
for name, build in {"one agent (steps)": lambda: build_single_agent_desk(offline_model(), tools, checkpointer=InMemorySaver()),
                    "agent subgraphs": lambda: build_handoffs_desk(offline_model, tools, checkpointer=InMemorySaver())}.items():
    meter = UsageMeter()
    report = evaluate(build(), list(by_id.values()), desk, meter=meter)
    rows.append({"form": name, "passed": f"{report['passed']}/15", "calls": meter.model_calls,
                 "tokens": f"{meter.total_tokens:,}",
                 "calls per agent": {k: v["model_calls"] for k, v in meter.agents().items()}})
table(rows, ["form", "passed", "calls", "tokens", "calls per agent"])

**You should now see** `15/15` and **53** calls for both forms (same decisions, same prompts), about 24K tokens each; the one-agent form shows all calls under `support_desk`, the subgraph form splits them as `triage: 16, billing: 21, technical: 10, account: 6`.

**Which form?** Start with the **one-agent** form: one graph, one state, one checkpoint, simplest to trace. Move to subgraphs when specialists need their own middleware, state or ownership. Compared with subagents (71 calls), handoffs save the coordinator's second call on every turn, but you lose the single place where every answer is checked.

---
## Part 6 · 💸 Handoffs with real models

> 💸 Calls `gpt-oss-20b` several times (multi-agent = many small calls), a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE: triage → specialists on gpt-oss-20b, a domain switch (S13) and a rejected refund (S03).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    def live_model(role):
        return get_model("fast", temperature=0, reasoning_effort="low")

    live = build_handoffs_desk(live_model, tools, checkpointer=InMemorySaver())
    meter = UsageMeter()
    report = evaluate(live, [by_id["S13"], by_id["S03"]], desk, meter=meter)
    for row in report["rows"]:
        o = row["outcome"]
        print(row["id"], "✅" if row["passed"] else "❌", o["route"], o["tools"], "pauses:", o["interrupts"], "|", o["answer"][:120])
    print(meter.agents())
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) S13 routed `['billing', 'technical']` with `lookup_invoices` and `search_faq`, and S03 with `pauses: 1` and an answer saying the refund was not approved; per-agent calls close to the offline ones. Real models sometimes answer a question themselves instead of transferring (watch `route`): tighten the triage prompt if they do. Without a key: `⏭️ Skipped`.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| Provider error like `tool_call_id ... has no response` / "messages with role tool must be a response to a preceding message with tool_calls" | A handoff dropped the AI message with the transfer call or its `ToolMessage`, **or** the model called a transfer **in parallel** with another tool: the jump leaves before the sibling's result exists, so its call is orphaned (and a side effect may have run with nobody seeing the result) | Always carry the calling AI message **and** a `ToolMessage` with its `tool_call_id`, and refuse a transfer that has sibling calls so the model retries it alone (`handoff_tool` does both; `mastery.agents.orphaned_tool_calls(messages)` finds the bad ids). The docs' handoff example assumes no parallel tool calls |
| `GraphRecursionError` in a handoff graph | Agents hand back and forth (lost context, or two prompts that both say "not mine") | `carry="turn"` or a summary in the transfer message; one clear owner per topic; always set `recursion_limit` |
| The second turn starts at triage again | `active_agent` / `current_step` not saved (no checkpointer, or a new `thread_id`) | Checkpointer on the outer graph, same `thread_id` per conversation |
| Specialist can't see its tools in the one-agent form | Tools not registered on `create_agent` | Register **all** tools; the middleware only filters |
| Triage answers the question itself | The model ignored "do not solve" | Give triage only transfer tools (no domain tools), keep its prompt short |
| Answer is "I'm passing this conversation to a human support agent" | The loop guard fired: more than `max_handoffs` transfers in one turn | Read the turn's messages: which agent kept transferring, and what did it not see? (Part 3) |

## ⚠️ Production notes

> ⚠️ **Production note: handoffs are a state machine; draw it.**
> List the allowed transitions (who can transfer to whom) and give each agent only those transfer tools. Unrestricted transfers are how ping-pongs start.

> ⚠️ **Production note: the active agent is user-visible state.**
> It persists across turns, so a user who changes topic is first answered by the old specialist. Make "transfer back to triage" possible, or reset `active_agent` after a resolved issue.

## 🏋️ Exercises

1. **(Easy)** Print `one_agent.get_state(config).values["current_step"]` after Part 1, then start a new `thread_id` and check that it starts at triage again.
2. **(Medium)** Add a `transfer_to_triage` tool to every specialist in Part 1 (`step_tool("triage")`) and a rule to the specialist prompt: "if the issue is resolved and the customer asks something unrelated, transfer to triage".
3. **(Medium)** Instead of `carry="turn"`, fix Part 3 by putting a **summary** of the tool results into the transfer `ToolMessage` content (write your own `handoff_tool`).
4. **(Stretch)** Restrict transitions: billing may transfer only to technical, technical only to account, account to nobody. Re-run the test set: which conversations fail now, and why?

<details>
<summary><b>Show solutions</b></summary>

**1.** `{"configurable": {"thread_id": "steps-2"}}` → the first model call uses the triage prompt again (state is per thread).

**2.**
```python
back = step_tool("triage", description="Send the customer back to triage for an unrelated new topic.")
for d in DOMAINS:
    steps[d]["tools"].append(back)
steps["triage"]["tools"] = list(transfers.values())
# register `back` on create_agent too: tools=[back, *transfers.values(), ...]
```

**3.**
```python
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.types import Command

def summary_handoff(target):
    @tool(f"transfer_to_{target}", description=f"Transfer to {target}.")
    def run(runtime: ToolRuntime) -> Command:
        msgs = runtime.state["messages"]
        last_ai = next(m for m in reversed(msgs) if isinstance(m, AIMessage))
        done = "; ".join(m.content for m in msgs if isinstance(m, ToolMessage) and not m.name.startswith("transfer"))
        note = ToolMessage(f"Transferred to {target}. Already done: {done or 'nothing'}", tool_call_id=runtime.tool_call_id)
        return Command(goto=target, graph=Command.PARENT, update={"active_agent": target, "messages": [last_ai, note]})
    return run
```
The offline model only counts real tool results, so test this with a live model (or extend the offline policy).

**4.** Pass only the allowed transfer tools per specialist (copy `build_handoffs_desk`'s loop and filter `others`). Run it: offline, **all 15 conversations still pass**, because none of them needs a forbidden transfer (S13 and S14 go billing → technical, which is allowed). That is the real lesson: a test set must contain the transitions you restrict. Add a conversation that does (a 429 question, then "also refund INV-1003 ($49)" in the next turn): technical can no longer hand the refund to billing, so the customer gets "can't help" or a wrong answer. Every restriction is a product decision; write it in your ADR.
</details>

## 🔎 Docs drill (5 min)

In the docs' handoff example, a transfer tool returns `Command(goto=..., graph=Command.PARENT)`. (1) What does `Command.PARENT` refer to, according to the API reference? (2) What does the handoffs guide assume about parallel tool calls?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. API reference for `langgraph.types.Command`: `graph=Command.PARENT` sends the command to the **closest parent graph** (here: the desk graph around the agent subgraph).
2. [Handoffs](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs), multiple agent subgraphs: the example assumes the model does **not** call a transfer in parallel with other tools. That's why `handoff_tool` refuses a transfer that has sibling calls (Pitfalls table).
3. In Jupyter: `Command??` shows the installed source and its docstring.

</details>

## ✅ Checkpoint

- [ ] I built handoffs in one agent with `current_step` + `@wrap_model_call` middleware.
- [ ] I built handoffs between agent subgraphs with `Command.PARENT` and `active_agent`.
- [ ] I can explain why repeat requests are cheaper with handoffs than with subagents.
- [ ] I reproduced the ping-pong bug, fixed it, and saw the loop guard escalate it.
- [ ] A refund over $50 paused inside a handoff and the rejection reached the customer.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read [Handoffs](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs) → *Context engineering* and note in [`docs/research-log.md`](../../docs/research-log.md): what the docs recommend passing to the next agent today (only the last message, a summary, or the full history), and whether that changed since 2026-10-12.

## Summary

- Handoffs = state decides who answers; the active agent persists across turns (no triage on repeat requests: 5 calls vs subagents' 8 for S12).
- One agent + middleware (`current_step`) is the simple, recommended form; agent subgraphs + `Command.PARENT` when specialists must be separate.
- Every handoff must keep the AI tool call and its `ToolMessage` together, and must carry (or summarize) the work done, or agents ping-pong.
- `interrupt()` works inside handoffs; the pause shows up as the specialist node.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 09.04 · Skills and routers** → [`09-04-skills-and-router.ipynb`](09-04-skills-and-router.ipynb).

[Module 09 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)